<a href="https://colab.research.google.com/github/slendrac123/I.A.-y-Mini-Robots/blob/main/2.%20Aut%C3%B3matas%20Celulares/Ejercicio%202.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Actividad 2: Modelo de Difusión – Simulación de Robot Evitando Obstáculos

Este cuaderno implementa una simulación interactiva de un **robot con dos ruedas** que navega en una cuadrícula 10×10, evitando obstáculos y buscando una celda objetivo.  
La simulación incluye **dos modelos de movimiento**:

- 🟡 **Modelo probabilístico de difusión (ACs probabilísticos)**  
- 🔵 **Modelo determinista con planificación (Algoritmo A\*)**


---

## Interfaz de Usuario y Controles

La interfaz está compuesta por:

- Una **cuadrícula 10×10 de botones** que representa el mapa del entorno.
- Cuatro botones de control:

**Controles:**

- 🟨 **Obstáculos (Amarillo):**  
  Permite alternar una celda entre **obstáculo** y **celda libre** al hacer clic.

- 🟩 **Inicio (Verde):**  
  Define la posición inicial del robot.

- 🟥 **Meta (Rojo):**  
  Define el objetivo al que debe llegar el robot.

- 🟦 **Simular (Azul):**  
  Inicia la simulación del robot usando el algoritmo seleccionado.

---

## Modelos de Movimiento del Robot

### 🔵 Algoritmo A* (Planificación Determinista)

En este modo, el robot utiliza el **algoritmo A\*** para encontrar el **camino más corto** desde el inicio hasta la meta, evitando obstáculos.

Características:

- El robot **conoce el mapa completo**.
- Usa una **heurística (distancia Manhattan)** para estimar el costo restante.
- Encuentra una ruta eficiente si existe.
- El movimiento se visualiza paso a paso en la cuadrícula.

Este enfoque representa un **agente con planificación global**.

---

### 🟡 Modelo Probabilístico de Difusión (Autómata Celular Estocástico)

En este modo, el robot se mueve de forma **probabilística**:

- En cada paso:
  - Observa sus **vecinos libres** (arriba, abajo, izquierda, derecha).
  - Elige **aleatoriamente** uno de ellos.
- No tiene conocimiento global del mapa.
- No sabe dónde está la meta (o solo la usa como condición de parada).
- Puede explorar el entorno, retroceder o quedar oscilando entre celdas.

Este comportamiento modela un:

> **Proceso de difusión estocástica sobre un autómata celular**, donde el robot se propaga por el espacio de manera local y probabilística.

Este modelo es útil para:

- Representar exploración sin planificación.
- Comparar contra algoritmos óptimos como A\*.
- Simular incertidumbre o agentes sin conocimiento global del entorno.

---

## Comparación de Enfoques

| Enfoque | Tipo | Conocimiento del mapa | Garantiza llegar | Eficiencia |
|--------|------|------------------------|------------------|------------|
| Difusión probabilística | Estocástico | Local | ❌ No | Baja |
| A* | Determinista | Global | ✅ Sí (si existe camino) | Alta |

---

## Estado Global del Mapa

Las variables globales gestionan la simulación:

- `N`: tamaño del mapa (10)  
- `grid`: matriz del entorno (0 = libre, 1 = obstáculo)  
- `start`: posición inicial  
- `goal`: posición objetivo  
- `robot`: posición actual del robot  
- `mode`: modo de edición (obstáculo, inicio, meta)  

---



In [22]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
import ipywidgets as widgets
from IPython.display import display
import time
import functools
import heapq
import random

# Estado global del mapa
N = 10
grid = np.zeros((N, N), dtype=int)   # 0 = libre, 1 = obstáculo
start = None
goal = None
robot = None
mode = "obstacle"  # "obstacle", "start", "goal"

In [23]:
def set_mode(m):
    global mode
    mode = m
    print("Modo actual:", mode)

In [24]:


button_obstacle = widgets.Button(description="Obstáculos", button_style='warning')
button_start = widgets.Button(description="Inicio", button_style='success')
button_goal = widgets.Button(description="Meta", button_style='danger')
button_simulate = widgets.Button(description="Simular", button_style='info')
button_probabilistic_simulate = widgets.Button(description="Simular Probabilístico", button_style='primary')

def on_button_obstacle_clicked(b):
    set_mode("obstacle")

def on_button_start_clicked(b):
    set_mode("start")

def on_button_goal_clicked(b):
    set_mode("goal")
def on_button_simulate_clicked(b):
    simulate(50)  # puedes cambiar 50 por más/menos pasos
def on_button_probabilistic_simulate_clicked(b):
    probabilistic_simulate(50) # You can change 50 to more/fewer steps

button_obstacle.on_click(on_button_obstacle_clicked)
button_start.on_click(on_button_start_clicked)
button_goal.on_click(on_button_goal_clicked)
button_simulate.on_click(on_button_simulate_clicked)
button_probabilistic_simulate.on_click(on_button_probabilistic_simulate_clicked)



In [25]:
def astar(grid, start, goal):
    N = grid.shape[0]

    def h(a, b):
        return abs(a[0]-b[0]) + abs(a[1]-b[1])  # heurística Manhattan

    open_set = []
    heapq.heappush(open_set, (0, start))

    came_from = {}
    g_score = {start: 0}

    while open_set:
        _, current = heapq.heappop(open_set)

        if current == goal:
            path = []
            while current in came_from:
                path.append(current)
                current = came_from[current]
            path.append(start)
            return path[::-1]

        for dx, dy in [(-1,0),(1,0),(0,-1),(0,1)]:
            nx, ny = current[0]+dx, current[1]+dy

            if 0 <= nx < N and 0 <= ny < N and grid[nx, ny] == 0:
                neighbor = (nx, ny)
                tentative_g = g_score[current] + 1

                if neighbor not in g_score or tentative_g < g_score[neighbor]:
                    came_from[neighbor] = current
                    g_score[neighbor] = tentative_g
                    f = tentative_g + h(neighbor, goal)
                    heapq.heappush(open_set, (f, neighbor))

    return None

In [26]:
def neighbors(pos):
    x, y = pos
    for dx, dy in [(-1,0),(1,0),(0,-1),(0,1)]:
        nx, ny = x+dx, y+dy
        if 0 <= nx < N and 0 <= ny < N and grid[nx, ny] == 0:
            yield (nx, ny)

def simulate(steps=100):
    global robot

    if start is None or goal is None:
        print("Define inicio y meta primero")
        return

    path = astar(grid, start, goal)

    if path is None:
        print("No hay camino posible")
        return

    for pos in path:
        robot = pos
        update_button_display() # Corrected call
        time.sleep(0.3)

    print("¡Llegó a la meta!")
    import random

def probabilistic_simulate(steps=100):
    global robot

    if start is None:
        print("Define un punto de inicio primero.")
        return

    robot = start
    update_button_display()
    time.sleep(0.3)

    for _ in range(steps):
        if robot == goal:
            print("¡Llegó a la meta!")
            break

        valid_neighbors = list(neighbors(robot)) # Use the existing neighbors function

        if not valid_neighbors:
            print("No hay más movimientos posibles desde la posición actual.")
            break

        next_pos = random.choice(valid_neighbors)
        robot = next_pos
        update_button_display()
        time.sleep(0.3)
    else:
        print("Simulación terminada después de", steps, "pasos.")



In [28]:
def update_button_display():
    for r in range(N):
        for c in range(N):
            button = buttons_grid[r][c]
            if robot == (r, c):
                button.style.button_color = 'blue'  # Robot
            elif goal == (r, c):
                button.style.button_color = 'red'   # Goal
            elif start == (r, c):
                button.style.button_color = 'green' # Start
            elif grid[r, c] == 1:
                button.style.button_color = 'gray'  # Obstacle
            else:
                button.style.button_color = 'lightgray' # Free

# Call the function to update the display initially
update_button_display()
print("update_button_display function defined and called for initial render.")

# Create a 2D list to store the buttons
buttons_grid = []
for r in range(N):
    row_buttons = []
    for c in range(N):
        button = widgets.Button(description=f'({r},{c})') # Placeholder description
        row_buttons.append(button)
    buttons_grid.append(row_buttons)

# Display the grid of buttons using GridspecLayout
grid_layout = widgets.GridspecLayout(N, N)
for r in range(N):
    for c in range(N):
        grid_layout[r, c] = buttons_grid[r][c]

display(grid_layout)
print("10x10 grid of ipywidgets.Button objects created and displayed.")
display(button_obstacle)
display(button_start)
display(button_goal)
display(button_simulate)
display(button_probabilistic_simulate)

def on_button_click(button_instance, r, c):
    global grid, start, goal, mode, buttons_grid # buttons_grid is needed to update other buttons' colors

    print(f"Clicked button at ({r},{c}) in mode: {mode}")

    if mode == "obstacle":
        grid[r, c] = 1 - grid[r, c]   # toggle obstacle
        if grid[r, c] == 1:
            button_instance.style.button_color = 'gray' # Obstacle
        else:
            button_instance.style.button_color = 'lightgray' # Free
    elif mode == "start":
        # If there was a previous 'start', reset its button's color
        if start is not None:
            prev_start_r, prev_start_c = start
            prev_start_btn = buttons_grid[prev_start_r][prev_start_c]
            # Only reset if the previous start is not also the current goal
            if (prev_start_r, prev_start_c) != goal:
                if grid[prev_start_r, prev_start_c] == 1: # If it was an obstacle
                    prev_start_btn.style.button_color = 'gray'
                else: # If it was free
                    prev_start_btn.style.button_color = 'lightgray'
        start = (r, c)
        button_instance.style.button_color = 'green' # New start
    elif mode == "goal":
        # If there was a previous 'goal', reset its button's color
        if goal is not None:
            prev_goal_r, prev_goal_c = goal
            prev_goal_btn = buttons_grid[prev_goal_r][prev_goal_c]
            # Only reset if the previous goal is not also the current start
            if (prev_goal_r, prev_goal_c) != start:
                if grid[prev_goal_r, prev_goal_c] == 1: # If it was an obstacle
                    prev_goal_btn.style.button_color = 'gray'
                else: # If it was free
                    prev_goal_btn.style.button_color = 'lightgray'
        goal = (r, c)
        button_instance.style.button_color = 'red' # New goal

# Initialize all buttons to 'lightgray' (assuming grid is initially all 0s - free)
for r in range(N):
    for c in range(N):
        buttons_grid[r][c].style.button_color = 'lightgray'

# Attach click handlers to all buttons
for r in range(N):
    for c in range(N):
        buttons_grid[r][c].on_click(functools.partial(on_button_click, r=r, c=c))

print("Button click logic implemented and handlers attached.")

update_button_display function defined and called for initial render.


GridspecLayout(children=(Button(description='(0,0)', layout=Layout(grid_area='widget001'), style=ButtonStyle()…

10x10 grid of ipywidgets.Button objects created and displayed.


Button(button_style='warning', description='Obstáculos', style=ButtonStyle())

Button(button_style='success', description='Inicio', style=ButtonStyle())

Button(button_style='danger', description='Meta', style=ButtonStyle())

Button(button_style='info', description='Simular', style=ButtonStyle())

Button(button_style='primary', description='Simular Probabilístico', style=ButtonStyle())

Button click logic implemented and handlers attached.
Modo actual: start
Clicked button at (5,1) in mode: start
¡Llegó a la meta!
Modo actual: start
Modo actual: obstacle
Clicked button at (7,4) in mode: obstacle
No hay camino posible
No hay camino posible
No hay camino posible
Modo actual: goal
Clicked button at (7,5) in mode: goal
¡Llegó a la meta!
Simulación terminada después de 50 pasos.
